# STEAM SCRAPING

Для парсинга в рамках нашей предметной области выбор пал на Steam, как на самую популярную площадку для покупки игр - занимает ~75% мирового рынка дистрибуции игр.

Кстати, для извлечения информации со стима есть Python-пакет steamcrawl и многие другие готовые решения, но мы не будем читерить и напишем свой скрапинг стима.

Сначала установим Chrome

- https://docs.cloud.google.com/architecture/chrome-desktop-remote-on-compute-engine

In [ ]:
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!sudo apt install ./google-chrome-stable_current_amd64.deb -y

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  119M  100  119M    0     0   339M      0 --:--:-- --:--:-- --:--:--  339M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 11.2 MB/136 MB of arc

Для имитации действий пользователя используем библиотеку Selenium. C помощью нее открывали браузер и страницы с играми, обрабатывали ожидания загрузки игр, подтверждали возраст для 18+ игр

- https://habr.com/ru/companies/otus/articles/596071/ - шпаргалка по Selenium
- https://www.testmuai.com/blog/selenium-wait-for-page-to-load/ - про обработку ожиданий в Selenium
- https://sqatools.in/chrome-options-in-python-selenium/ - options для управления поведением браузера в selenium

By - для поиска элементов на странице

С помощью bs4 парсили html - BeautifulSoup делает это быстрее Selenium

In [ ]:
!pip3 install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
import selenium
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

Логирование осуществили через файл настроек logging_config.ini, сами логи лежат в файле steam_parser, где можно посмотреть как проходил парсинг (оба файла лежат в папке logs)

- https://habr.com/ru/articles/513966/
- https://docs.python.org/3/howto/logging.html#configuring-logging

In [ ]:
import logging
from logging.config import fileConfig

import os

In [ ]:
os.makedirs('logs', exist_ok=True)

fileConfig('./logs/logging_config.ini')

Для более эффективного парсинга импользуем httpx для асинхронных запросов.
Selenium использует окно браузера, а httpx - http запросы, которые можно делать много и одновременно и таким образом экономить ресурсы.

- https://habr.com/ru/companies/ru_mts/articles/968776/ - про инструменты для ускорения производительности
- https://www.python-httpx.org/ - httpx дока
- https://habr.com/ru/articles/667630/ - asyncio

In [ ]:
!pip install httpx

Каждую игру будем хранить в объекте классе SteamGame, внутри которого определим методы получения различных полей асинхронно с помощью клиента-httpx и selenium. Эти классы будут использоваться как контейнеры для данных, обеспечивая строгую типизацию

- https://habr.com/ru/articles/902634/ - освежить знания ООП

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

In [ ]:
@dataclass
class SteamReviewsSummary:
    total_reviews: int = 0
    rating_value: int = 0
    best_rating: int = 0
    worst_rating: int = 0

@dataclass(frozen=True)
class SteamGame:
    title: str = ""
    release_date: str = ""
    developers: list[str] = field(default_factory=list)
    additional_info: dict[str] = field(default_factory=dict)
    description: str = ""

    price: Optional[float] = None
    currency: str = "USD"

    tags: List[str] = field(default_factory=list)
    reviews: SteamReviewsSummary = field(default_factory=SteamReviewsSummary)

Парсинг списка игр идет через постановку асинхронных задач  - задачи выполняются через http-запрос, если он не работает (например, если требуется подтверждение возраста, или Steam посчитал запрос подозрительным, или он перегружен), то через Selenium

- https://www.geeksforgeeks.org/python/queue-in-python/ - очередь для хранения и переиспользования драйверов
- https://habr.com/ru/companies/simbirsoft/articles/701020/ - работа с потоками

In [ ]:
from queue import Queue
from selenium.webdriver import Chrome
from threading import Lock

Реализуем пул Chrome-драйверов для многопоточного парсинга, где при инициализации создается заданное количество экземпляров браузера, которые отправляются в очередь

Метод get_driver забирает драйвер из очереди и увеличивает счетчик активных драйверов под защитой блокировки Lock

Затем вызывается release_driver, который очищает все куки для изоляции сессий, уменьшает счетчик активных драйверов и возвращает браузер обратно в пул для повторного использования

При завершении работы метод close_all проходит по всем драйверам в очереди и завершает их процессы через quit. Переиспользование драйверов дает выигрыш в скорости при парсинге сотен страниц.



In [ ]:
class WebDriverPool:
    def __init__(self, max_drivers=3, options=None):
        self.max_drivers = max_drivers
        self.options = options or Options()
        self._pool = Queue(maxsize=max_drivers)
        self._lock = Lock()
        self._active_count = 0

        for _ in range(max_drivers):
            driver = webdriver.Chrome(self.options)
            self._pool.put(driver)

    def get_driver(self):
        driver = self._pool.get()
        with self._lock:
            self._active_count += 1
        return driver

    def release_driver(self, driver):
        driver.delete_all_cookies()
        with self._lock:
            self._active_count -= 1
        self._pool.put(driver)

    def close_all(self):
        while not self._pool.empty():
            driver = self._pool.get()
            driver.quit()

    def get_active_count(self):
        with self._lock:
            return self._active_count

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

import re, httpx, random

Парсинг списка игр идет через постановку асинхронных задач  - задачи выполняются через http-запросы, если они не работают (например, если требуется подтверждение возраста, или Steam посчитал запрос подозрительным), то через Selenium.



In [ ]:
class SteamParser:
    def __init__(self, web_driver_pool=None, log_file='steam_parser.log', log_level=logging.INFO, sleep_parse_game=5,
                 max_httpx_concurrent=5, request_delay=0.5):
        # настройка драйверов, логгера, семафора, задержек между запросами и пауз между парсингом игр
        self.web_driver_pool = web_driver_pool

        self.sleep_parse_game = sleep_parse_game

        self.options = Options()

        prefs = {"profile.managed_default_content_settings.images": 2}
        self.options.add_experimental_option("prefs", prefs)

        self.options.page_load_strategy = 'eager'

        self.options.add_argument('--no-sandbox')
        self.options.add_argument('--disable-dev-shm-usage')
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")


        if self.web_driver_pool is None:
            self.web_driver_pool = WebDriverPool(max_drivers=3, options=self.options)

        self.logger = logging.getLogger('steam_parser')
        self.httpx_semaphore = asyncio.Semaphore(max_httpx_concurrent)
        self.request_delay = request_delay
        self._last_request_time = 0

        self.request_count = 0
        self.error_count = 0
        self.start_time = time.time()

    def parse_games_links(self, page: int) -> list[str]:
        """
        парсинг ссылок игр со страницы без ассинхронности
        """

        self.logger.info(f"processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page={page}")

        driver = self.web_driver_pool.get_driver()

        driver.get(f"https://store.steampowered.com/search/?ignore_preferences=1&page={page}")

        html = driver.page_source

        soup = BeautifulSoup(html, 'html.parser')

        allLinks = []

        items = soup.find_all('div', id='search_resultsRows')
        for item in items:
            links = item.find_all('a')
            for link in links:
                url = link.get('href')
                if url:
                    allLinks.append(url.split('?')[0].rstrip('/'))

        self.web_driver_pool.release_driver(driver)

        return allLinks


    async def get_content_via_httpx(self, link: str, client: httpx.AsyncClient) -> list[BeautifulSoup, bool]:
        """
        Асинхронно загружает страницу через httpx с ретраями и семафором.
        При успехе возвращает soup, при возрастном ограничении фгаг использовать Selenium - True

        """
        async with self.httpx_semaphore:
            try:
                    response = await self._fetch_with_retry(client, link, max_retries=5) # здесь для ретраев и наждежной обработк игры

                    if response is None:
                        self.logger.error(f"Failed to fetch link after retries: {link}")
                        return None, False

                    if response.status_code != 200:
                        self.logger.error(f"Failed to fetch link {link}: Status {response.status_code}")
                        return None, False

                    html = response.text
                    soup = BeautifulSoup(html, 'html.parser')

                    if not soup.find('div', id='appHubAppName') and soup.find('div', class_='age_gate'):
                        self.logger.info(f"Age check detected via requests for {link}")
                        return None, True

                    return soup, False
            except Exception as e:
                self.logger.error(f"httpx.request error: {e}")
                self.logger.error(f"httpx error type: {type(e)}, args: {e.args}")
            return None, False


    async def parse_game_links_v2(self, page: int, client: httpx.AsyncClient) -> list[str]:
        """
        Ассинхронный парсинг ссылок игр со страницы

        """
        url = f"https://store.steampowered.com/search/?ignore_preferences=1&page={page}"

        self.logger.info(f"Processing parsing game links from {url}")

        try:
                response = await client.get(url)

                if response.status_code != 200:
                    self.logger.error(f"Failed to fetch page {page}: Status {response.status_code}")
                    return []

                html = response.text
                soup = BeautifulSoup(html, 'html.parser')
                all_links = []

                search_results = soup.find('div', id='search_resultsRows')
                if not search_results:
                    self.logger.warning(f"No search results found on page {page}")
                    return []

                links = search_results.find_all('a')
                for link in links:
                    url = link.get('href')
                    if url:
                        url = url.split('?')[0].rstrip('/')
                        all_links.append(url)

                return all_links

        except Exception as e:
            self.logger.error(f"async parsing error on page {page}: {e}")
            return []
        finally:
            await asyncio.sleep(5)


    def get_soup_via_selenium(self, link: str):
        """
        Подтверждение возраста

        """
        self.logger.info(f"using Selenium for {link}")

        driver = self.web_driver_pool.get_driver()

        driver.get(link)

        try:
            age_select = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "ageYear"))
            )

            from selenium.webdriver.support.ui import Select
            Select(age_select).select_by_value("2000")

            view_page_btn = driver.find_element(By.ID, "view_product_page_btn")
            view_page_btn.click()

            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.ID, "appHubAppName"))
            )
        except Exception as e:
            self.logger.error(f"get_soup_via_selenium error {e}")
        finally:
            self.web_driver_pool.release_driver(driver)

        return BeautifulSoup(driver.page_source, 'html.parser')


    def parse_game(self, link: str) -> SteamGame:
        """
        Синхронный парсинг атрибутов игры

        """
        driver = self.web_driver_pool.get_driver()
        driver.get(link)

        text = driver.page_source
        soup = BeautifulSoup(text, 'html.parser')

        try:
            price, currency = self.parse_game_price(soup, link)

            parsed_game = SteamGame(
                title=self.parse_name_game(soup, link),
                release_date=self.parse_release_date_game(soup),
                tags=self.parse_tags(soup),
                price=price,
                currency=currency,
                description=self.parse_game_description(soup),
                developers=self.parse_developers(soup),
                reviews=self.parse_reviews_summary(soup),
                additional_info=self.parse_additional_description(soup),
            )
            return parsed_game
        except Exception as e:
            self.logger.info(f"exception game processing {e}, location {e.__traceback__}")
        finally:
            self.web_driver_pool.release_driver(driver)

        return None


    async def parse_game_v2(self, link: str, client: httpx.AsyncClient) -> SteamGame:
        """
        Асинхронный парсинг игры

        """
        result_httpx, neededToSelenium = await self.get_content_via_httpx(link, client)
        if result_httpx is None:
            self.logger.warning(f"game on {link} failed to fetch info, trying to selenium is {neededToSelenium}")
            soup = self.get_soup_via_selenium(link)
        else:
            soup = result_httpx

        try:
            price, currency = self.parse_game_price(soup, link)

            parsed_game = SteamGame(
                title=self.parse_name_game(soup, link),
                release_date=self.parse_release_date_game(soup),
                tags=self.parse_tags(soup),
                price=price,
                currency=currency,
                description=self.parse_game_description(soup),
                developers=self.parse_developers(soup),
                reviews=self.parse_reviews_summary(soup),
                additional_info=self.parse_additional_description(soup),
            )


            return parsed_game
        except Exception as e:
            self.logger.info(f"exception game processing {e}, location {e.__traceback__}")
        finally:
            delay = self.sleep_parse_game + random.uniform(0, 2)
            await asyncio.sleep(delay)

        return None



    def parse_tags(self, soup):

        result = []
        tags = soup.find('div', class_='glance_tags popular_tags')
        for tag in tags.find_all('a'):
            result.append(tag.text.strip())
        return result

    def float_chars(self, s: str):
      """
      проверяет является ли символ цифрой или запятой
      используем для фильтрации строки цены при очистке от лишних символов

      """
      return str.isdigit(s) or s == ","


    def parse_currency(self, text: str) -> str:
      """
      Определяет валюту по символу или сокращению в тексте цены
      По умолчанию USD

      """
      currency_map = {
          '₽': 'RUB',
          'руб.': 'RUB',
          'rub': 'RUB',
          'р.': 'RUB',
          'nt$': 'TWD',
          '$': 'USD',
          'usd': 'USD',
          '€': 'EUR',
          'eur': 'EUR',
          '£': 'GBP',
          'gbp': 'GBP',
          '¥': 'JPY',
          'jpy': 'JPY',
          '₩': 'KRW',
          'krw': 'KRW',
          'rmb': 'CNY',
          '¥': 'CNY'
      }
      currency = "USD"
      for symbol, curr in currency_map.items():
          if symbol in text:
              currency = curr
              break
      return currency


    def parse_game_price(self, soup, link: str) -> list[float, str]:
        try:
            price_elem = soup.find('div', class_='game_purchase_price price')
            if price_elem is None:
                price_elem = soup.find('div', class_='discount_final_price')

            price_elem = soup.find('div', class_='game_purchase_price price').text.strip()
            if "free" in price_elem.lower():
              return 0, "USD"

            currency = self.parse_currency(price_elem.lower())

            cleaned_price = re.sub(r'[^\d,.]', '', price_elem)

            if ',' in cleaned_price and '.' in cleaned_price:
              cleaned_price = cleaned_price.replace(',', '')
            elif ',' in cleaned_price:
              if len(cleaned_price.split(',')[-1]) == 2:
                cleaned_price = cleaned_price.replace(',', '.')
              else:
                cleaned_price = cleaned_price.replace(',', '')

            if cleaned_price == "":
              return 0, "USD"

            return float(cleaned_price), currency
        except Exception as e:
            self.logger.info(f"game {link}, has no price exception {e}")

            return 0, "USD"

    def parse_additional_description(self, soup) -> list[str]:
        try:
            content_descriptors = soup.find('div', id='game_area_content_descriptors')
            return content_descriptors.text.strip()
        except Exception as e:
            self.logger.info(f"parse_additional_description exception due {e}")
            return ""


    def parse_name_game(self, soup: BeautifulSoup, link) -> str:
        try:
            name = soup.find('div', id='appHubAppName')
            return name.text.strip()
        except AttributeError:
            self.logger.error(f"game {link} has wrong name tag")

            return ""

    def parse_release_date_game(self, soup: BeautifulSoup) -> str:
        try:
            release_date_elem = soup.find('div', class_='release_date')
            return release_date_elem.find('div', class_='date').text.strip()
        except Exception as e:
            self.logger.error(f"parse_release_date_game exception due {e}")
            return ""

    def parse_games_on_links(self, links: list[str]) -> list[SteamGame]:
        """
        Парсим список игр

        """
        res = []
        for i in links:
            res_parse = self.parse_game(i)
            if res_parse is not None:
                res.append(res_parse)
        return res

    async def parse_games_on_links_v2(self, links: list[str], client) -> list[SteamGame]:
        """
        Ассинхронный парсинг игр из списка

        """
        tasks = [self.parse_game_v2(link, client) for link in links]

        results = await asyncio.gather(*tasks)

        return [game for game in results if game is not None and not isinstance(game, Exception)]

    def parse_game_description(self, soup: BeautifulSoup) -> str:
        try:
            description_elem = soup.find('div', class_='game_description_snippet')
            return description_elem.text.strip()
        except Exception as e:
            self.logger.error(f"exception due {e}")
            return ""

    def parse_reviews_summary(self, soup: BeautifulSoup) -> SteamReviewsSummary:
        """
        Возвращает класс SteamReviewsSummary

        """
        user_reviews_elem = soup.find('div', class_='user_reviews')
        reviews_aggr = user_reviews_elem.find('a',attrs={'itemprop': "aggregateRating"})

        def get_meta(prop):
            tag = reviews_aggr.find('meta', itemprop=prop)
            return tag.get('content', 0) if tag else 0

        return SteamReviewsSummary(
            total_reviews=int(get_meta('reviewCount')),
            rating_value=int(get_meta('ratingValue')),
            best_rating=int(get_meta('bestRating')),
            worst_rating=int(get_meta('worstRating')),
            )

    #словарь с http-заголовками для запросов,
    #имитирующими реальный браузер
    #используем для обхода базовой защиты и уменьшения вероятности блокировки при парсинге через httpx
    def _get_random_headers(self) -> dict:
        """
        Генерирует заголовки для запроса

        """
        return {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://steampowered.com",
            "Accept-Encoding": "gzip, deflate, br",
            "DNT": "1",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "none",
            "Sec-Fetch-User": "?1",
            "Cache-Control": "max-age=0",
        }

    async def _rate_limit(self):
        """
        Ограничение частоты запросов

        """
        now = time.time()
        elapsed = now - self._last_request_time
        if elapsed < self.request_delay:
            delay = self.request_delay - elapsed
            await asyncio.sleep(delay)
        self._last_request_time = time.time()

    async def _fetch_with_retry(self, client: httpx.AsyncClient, url: str,
                               max_retries: int = 3, base_delay: float = 1.0) -> Optional[httpx.Response]:
        """

        http-запрос с автоматическими повторными попытками при ошибках

        """
        last_exception = None

        for attempt in range(max_retries):
            try:
                await self._rate_limit()

                headers = self._get_random_headers()

                response = await client.get(
                    url,
                    headers=headers,
                    timeout=httpx.Timeout(15.0, connect=10.0, read=30.0),
                    follow_redirects=True
                )

                self.request_count += 1

                if response.status_code in [429, 500, 502, 503, 504]:
                    wait_time = base_delay * (2 ** attempt) + random.uniform(0, 1)
                    self.logger.warning(f"Server error {response.status_code} for {url}. "
                                       f"Retry {attempt+1}/{max_retries} in {wait_time:.2f}s...")
                    await asyncio.sleep(wait_time)
                    continue

                if "Access Denied" in response.text or "captcha" in response.text.lower():
                    self.logger.warning(f"Access denied or captcha detected for {url}")
                    wait_time = base_delay * (2 ** attempt) + random.uniform(2, 5)
                    await asyncio.sleep(wait_time)
                    continue

                return response

            except (httpx.ConnectTimeout, httpx.ReadTimeout, httpx.ConnectError,
                    httpx.PoolTimeout, httpx.RemoteProtocolError) as e:
                last_exception = e
                self.error_count += 1

                if attempt < max_retries - 1:
                    wait_time = base_delay * (2 ** attempt) + random.uniform(0, 2)
                    self.logger.warning(f"Request failed ({type(e).__name__}) for {url}. "
                                       f"Retry {attempt+1}/{max_retries} in {wait_time:.2f}s...")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Request failed after {max_retries} attempts for {url}: {e}")

            except Exception as e:
                self.logger.error(f"Unexpected error for {url}: {type(e).__name__}: {e}")
                self.error_count += 1
                if attempt == max_retries - 1:
                    raise

        return None


    def parse_developers(self, soup: BeautifulSoup) -> list[str]:
        description_elem = soup.find('div', id="developers_list")

        result = []
        for dev in description_elem.find_all('a'):
            result.append(dev.text.strip())
        return result

Теперь попробуем получить информию об одной игре

In [ ]:
SteamParser().parse_game("https://store.steampowered.com/app/730")

SteamGame(title='Counter-Strike 2', release_date='Aug 21, 2012', developers=['Valve'], additional_info='Mature Content Description\nThe developers describe the content like this:\n\n\t\t\tIncludes intense violence and blood.', description='For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.', price=0, currency='USD', tags=['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'eSports', 'Tactical', 'First-Person', 'PvP', 'Online Co-Op', 'Co-op', 'Strategy', 'Military', 'War', 'Difficult', 'Trading', 'Realistic', 'Fast-Paced', 'Moddable'], reviews=SteamReviewsSummary(total_reviews=2516257, rating_value=9, best_rating=10, worst_rating=1))

Перед непосредственным парсингом игр, добудем нужное количество ссылок на игры

Создаем 430 задач под каждую страницу, каждая вызывает parse_game_links_v2 для своей страницы

Семафор пропускает одновременно только 3 задачи, остальные ждут

httpx выполняет http-запросы, получает html, BeautifulSoup парсит ссылки на игры. Результаты собираются в список all_our_links

In [ ]:
async def worker_links(parse_links_func, page_num, client, semaphore):
    async with semaphore:
        res = await parse_links_func(page_num, client)
        print(f"processed page {page_num}")
        return res


all_our_links = []

custom_options = Options()

prefs = {"profile.managed_default_content_settings.images": 2}
custom_options.add_experimental_option("prefs", prefs)

custom_options.page_load_strategy = 'eager'

custom_options.add_argument('--no-sandbox')
custom_options.add_argument('--disable-dev-shm-usage')
custom_options.add_argument("--headless")
custom_options.add_argument("--disable-gpu")

web_driver_pool = WebDriverPool(max_drivers=4, options=custom_options)

async def main():
    limits = httpx.Limits(max_keepalive_connections=100, max_connections=150, keepalive_expiry=5.0,
        )
    timeout = httpx.Timeout(20.0, read=30.0)
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://steampowered.com"
    }

    async with httpx.AsyncClient(limits=limits, follow_redirects=True, timeout=timeout) as client:
        pages_count = 430
        limit = 50

        sem = asyncio.Semaphore(3)

        tasks = [worker_links(SteamParser(web_driver_pool=web_driver_pool).parse_game_links_v2, i, client, sem) for i in range(pages_count)]

        global all_our_links
        all_our_links = await asyncio.gather(*tasks)

await main()

INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=0
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=1
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=2
processed page 1
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=3
processed page 2
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=4
processed page 0
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=5
processed page 3
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page=6
processed page 4
processed page 5
INFO     - Processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&p

In [ ]:
len(all_our_links)

430

In [ ]:
all_our_links

[['https://store.steampowered.com/app/730/CounterStrike_2',
  'https://store.steampowered.com/app/230410/Warframe',
  'https://store.steampowered.com/app/3321460/Crimson_Desert',
  'https://store.steampowered.com/app/2868840/Slay_the_Spire_2',
  'https://store.steampowered.com/app/2767030/Marvel_Rivals',
  'https://store.steampowered.com/app/1172470/Apex_Legends',
  'https://store.steampowered.com/app/236390/War_Thunder',
  'https://store.steampowered.com/app/3564740/Where_Winds_Meet',
  'https://store.steampowered.com/app/1808500/ARC_Raiders',
  'https://store.steampowered.com/app/359550/Tom_Clancys_Rainbow_Six_Siege',
  'https://store.steampowered.com/app/2050650/Resident_Evil_4',
  'https://store.steampowered.com/app/3240220/Grand_Theft_Auto_V_Enhanced',
  'https://store.steampowered.com/app/252490/Rust',
  'https://store.steampowered.com/app/1151340/Fallout_76',
  'https://store.steampowered.com/app/306130/The_Elder_Scrolls_Online',
  'https://store.steampowered.com/app/578080/PUBG

переведем двумерный массив в одномерный

In [ ]:
links_to_parse = [link for sublist in all_our_links for link in sublist]
len(links_to_parse)

10525

Взяли 430 страниц с запасом (получилось 10525 игр, примерно по 25 игр на страницу) - на случай если какие-то игры не получится собрать или будут дубликаты

ссылки собирались ~ 14 минут

На всякий случай сохраним полученные ссылки в файл, чтобы не пришлось собирать их заново

Далее наконец приступим к парсингу игр с собранных страниц

In [ ]:
import csv

with open('./links.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['link'])
    writer.writeheader()
    for i in links_to_parse:
        writer.writerow({"link": i})

 Разбиваем список ссылок на пакеты фиксированного размера для последующей обработки


In [ ]:
proccessing_games = links_to_parse[:10525]

packets_games = []
cur_pak = []

threshold_packet = 1000

for i in proccessing_games:
    if len(cur_pak) >= threshold_packet:
        packets_games.append(cur_pak)

        cur_pak = [i]
    else:
        cur_pak.append(i)
packets_games.append(cur_pak)

In [ ]:
len(packets_games)

11

Теперь самое интересное - запускаем парсинг на 10к+ игр

In [ ]:
async def worker_parse_games(parse_games_func, links, client, semaphore):
    async with semaphore:
        res = await parse_games_func(links, client)
        print(f"processed games {len(links)}")
        return res


all_our_games = []

web_driver_pool = WebDriverPool(max_drivers=4, options=custom_options)
sem = asyncio.Semaphore(5)
all_our_games = []
tasks = []

async def main():
    headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://steampowered.com"
        }

    limits=httpx.Limits(
            max_connections=200,
            max_keepalive_connections=100,
            keepalive_expiry=5.0,
        )
    timeout = httpx.Timeout(20.0, read=30.0)

    async with httpx.AsyncClient(limits=limits, headers=headers, timeout=timeout, follow_redirects=True) as client:
        tasks = []
        for games in packets_games:
            coro = worker_parse_games(SteamParser(web_driver_pool, sleep_parse_game=2).parse_games_on_links_v2, games, client, sem)
            tasks.append(coro)

        results = await asyncio.gather(*tasks, return_exceptions=True)
        for res in results:
            if not isinstance(res, Exception):
                all_our_games.extend(res)

await main()

In [ ]:
len(all_our_games)

10463

Ура нас не забанили и связь не оборвалась

Получили информацию о 10463 играх

Парсинг занял около 3 часов 20 минут

In [ ]:
import pandas as pd
from collections import Counter


df = pd.DataFrame([vars(u) for u in all_our_games if u])

df

,title,release_date,developers,additional_info,description,price,currency,tags,reviews
0,Counter-Strike 2,"Aug 21, 2012",[Valve],Mature Content Description\nThe developers des...,"For over two decades, Counter-Strike has offer...",0.00,USD,"[FPS, Shooter, Multiplayer, Competitive, Actio...","SteamReviewsSummary(total_reviews=2516281, rat..."
1,Warframe,"Mar 25, 2013",[Digital Extremes],Mature Content Description\nThe developers des...,Awaken as an unstoppable warrior and battle al...,0.00,USD,"[Free to Play, Looter Shooter, Action RPG, Thi...","SteamReviewsSummary(total_reviews=290411, rati..."
2,Crimson Desert,"Mar 19, 2026",[Pearl Abyss],AI Generated Content Disclosure\nThe developer...,Crimson Desert is an open-world action-adventu...,69.99,USD,"[Open World, Action, Singleplayer, Adventure, ...","SteamReviewsSummary(total_reviews=44362, ratin..."
3,Slay the Spire 2,"Mar 5, 2026",[Mega Crit],,The iconic roguelike deckbuilder returns. Craf...,24.99,USD,"[Strategy, Roguelike, Card Game, Deckbuilding,...","SteamReviewsSummary(total_reviews=42097, ratin..."
4,Marvel Rivals,"Dec 5, 2024",[NetEase Games],,Marvel Rivals is a Super Hero Team-Based PVP S...,0.00,USD,"[Free to Play, Multiplayer, Hero Shooter, Thir...","SteamReviewsSummary(total_reviews=274971, rati..."
...,...,...,...,...,...,...,...,...,...
10458,PAYDAY 2: Dragan Character Pack,"Jan 22, 2015","[Lion game Lion, OVERKILL - a Starbreeze Studio.]",,,1.99,USD,"[Action, RPG]","SteamReviewsSummary(total_reviews=465, rating_..."
10459,HUNTER×HUNTER NEN×IMPACT,"Jul 16, 2025",[Bushiroad Games],,"""HUNTER x HUNTER""'s first full-fledged fightin...",59.99,USD,"[Action, 2D Fighter, PvP, Multiplayer, Comic B...","SteamReviewsSummary(total_reviews=219, rating_..."
10460,LobotomyCorporation_ArtBook,"Jul 13, 2018",[Project Moon],,,9.99,USD,"[Indie, Simulation, Management, Psychological ...","SteamReviewsSummary(total_reviews=193, rating_..."
10461,Helltaker: Artbook + Pancake Recipe,"May 11, 2020",[vanripper],,,9.99,USD,"[Indie, Free to Play, Adventure, Demons, Anime...","SteamReviewsSummary(total_reviews=1052, rating..."


In [ ]:
df['title'].duplicated().sum()

np.int64(52)

Даже дубликатов не много - всего 5% и пропущенных игр из ссылок только 62

Сохраним результат нашей работы в файл и перейдем к EDA

In [ ]:
import csv
import os

def object_to_dict(obj):
    result = {}
    for key, value in obj.__dict__.items():
        if hasattr(value, "__dict__"):
            nested = object_to_dict(value)
            for n_key, n_value in nested.items():
                result[f"{key}_{n_key}"] = n_value
        else:
            result[key] = value
    return result

steam_games_parsed = [object_to_dict(u) for u in all_our_games]
fields = steam_games_parsed[0].keys()

with open('./new_steam_parse.csv', 'w', newline='\n', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(steam_games_parsed)